In [ ]:
# pip install packages that are not in Pyodide
%pip install ipympl==0.9.3
%pip install seaborn==0.12.2

In [3]:
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import seaborn as sns
from cycler import cycler
%matplotlib widget

# Set the color scheme
sns.set_theme()
colors = [
    "#0076C2",
    "#EC6842",
    "#A50034",
    "#009B77",
    "#FFB81C",
    "#E03C31",
    "#6CC24A",
    "#EF60A3",
    "#0C2340",
    "#00B8C8",
    "#6F1D77",
]
plt.rcParams["axes.prop_cycle"] = cycler(color=colors)

# Predicting Pressure in a Water Network Using an MLP

In the last chapter we introduced a **Multilayer Perceptron (MLP)**. We will show an example of a Multilayer Perceptron (MLP) to estimate the nodal pressures from the BakRyan water distribution system as shown in the figure below.





```{figure} https://files.mude.citg.tudelft.nl/BAK.png

---

---
Numerical results for the BakRyan water distribution system
```



This system has 58 pipes and 35 nodes. We will go through the different steps of how to create and train an Artificial Neural Network, as shown in the figure below. Furthermore, we will hyper-optimize the MLP to improve its performance

```{figure} https://files.mude.citg.tudelft.nl/ANN_image2.png

---

---
Artificial neural network representation, with a zoomed-in view of how a single neuron works. For this notebook, we have 58 input features (pipe diameters) and 35 outputs (nodal pressures).
```

### Mathematical Formulation


 **Input:** A vector of pipe diameters  
 **Output:** A vector of nodal pressures

We can express the model as:

$$
\mathbf{y} = \phi(\mathbf{x}, \mathbf{W})
$$

Where:

- **$\mathbf{y}$**: output data (nodal pressures, units: *mwc*)
- **$\phi$**: represents the Artificial Neural Network
- **$\mathbf{x}$**: input data (pipe diameters, units: *m*)
- **$\mathbf{W}$**: parameters of the MLP (unitless)



Having pairs of input-output data (**diameters** $\mathbf{x}$ and **pressures** $\mathbf{y}$) the goal is to find the parameters $\mathbf{W}$ that best fit the data.




In [6]:
import os
from urllib.request import urlretrieve

def findfile(fname):
    if not os.path.isfile(fname):
        print(f"Downloading {fname}...")
        urlretrieve('http://files.mude.citg.tudelft.nl/'+fname, fname)

findfile('targets_BAK.pk')
findfile('features_BAK.pk')



In [7]:
file_path = r"features_BAK.pk" 
with open(file_path, 'rb') as handle:
    features = pickle.load(handle)

file_path = r"targets_BAK.pk"
with open(file_path, 'rb') as handle:
    targets = pickle.load(handle)

print('Dimensions of features (X):', features.shape)
print('Dimensions of targets  (t):', targets.shape)

Dimensions of features (X): (10000, 58)
Dimensions of targets  (t): (10000, 35)


/var/folders/9f/qj38cd3d5v37nszw4dsn87nr0000gn/T/ipykernel_73384/1801784045.py:3: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  features = pickle.load(handle)
/var/folders/9f/qj38cd3d5v37nszw4dsn87nr0000gn/T/ipykernel_73384/1801784045.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world

In [ ]:

X = np.loadtxt("ml/data/features_BAK.csv", delimiter=",")
y = np.loadtxt("ml/data/targets_BAK.csv", delimiter=",")

print(f"X shape: {X.shape}, y shape: {y.shape}")

X shape: (10000, 58), y shape: (10000, 35)
